# Training Binding Affinity Models with Google's TabFM zero-shot model.

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from tabfm import TabFMRegressor, TabFMClassifier, tabfm_v1_0_0_pytorch as tabfm_v1_0_0

In [2]:
reg_df = pd.read_csv("/scratch/cford38/savant_tests/combined_50k_filtered_prepped_regression.csv")

reg_df.head()

,molecular_weight,logP,num_h_donors,num_h_acceptors,num_rotatable_bonds,tpsa,num_rings,num_aromatic_rings,num_atoms,num_heavy_atoms,formal_charge,score,total,vdw,elec,desolv,bsa,num_interacting_residues,neg_log10_affinity_M
0,312.538,7.1127,1,2,19,37.30,0,0,62,22,0,-89.935,-218.023,-48.388,-169.635,-7.620,830.568,22,7.585027
1,406.438,4.8204,3,6,11,104.82,4,3,52,30,0,-47.412,-51.750,-45.487,-6.263,-0.672,752.575,19,7.638272
2,520.563,-1.8802,7,9,18,222.13,2,0,71,37,1,-80.642,-197.802,-44.507,-153.295,-5.476,989.693,27,5.191789
3,569.524,-4.5727,7,13,12,253.76,4,1,71,40,0,-29.119,-3.597,-31.511,27.915,-3.190,615.462,18,6.732828
4,322.408,3.0534,2,4,6,61.69,3,2,46,24,0,-53.062,-101.627,-34.512,-67.115,-5.127,622.828,18,4.565431


In [3]:
## Create the training and testing datasets
X = reg_df.drop(columns=['neg_log10_affinity_M'])
y = reg_df['neg_log10_affinity_M']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=1337
)

In [4]:
## Train the TabFM regression model
model = tabfm_v1_0_0.load(model_type="regression", device="cuda")
reg = TabFMRegressor(model=model)
reg.fit(X_train, y_train)

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights from local directory


TabFMRegressor(model=TabFM_HF(
  (cell_embedder): CellEmbedder(
    (in_linear): Linear(in_features=64, out_features=256, bias=True)
    (in_linear_cat): Linear(in_features=64, out_features=256, bias=True)
    (y_embedder_lookup): MLP(
      (layers): ModuleList(
        (0): Linear(in_features=1, out_features=6, bias=True)
        (1): Linear(in_features=6, out_features=256, bias=True)
      )
    )
  )
  (col_embedder): C...
          (linear2): Linear(in_features=8192, out_features=2048, bias=True)
        )
      )
    )
    (ln): RMSNorm()
    (y_encoder): MLP(
      (layers): ModuleList(
        (0): Linear(in_features=1, out_features=4096, bias=True)
        (1): Linear(in_features=4096, out_features=2048, bias=True)
      )
    )
    (decoder): MLP(
      (layers): ModuleList(
        (0): Linear(in_features=2048, out_features=4096, bias=True)
        (1): Linear(in_features=4096, out_features=1, bias=True)
      )
    )
  )
))

In [ ]:
## Generate Predictions

preds = reg.predict(X_test)

In [7]:
## Evaluate the regression model
import math
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error, r2_score

rmse = math.sqrt(mean_squared_error(y_test, preds))
mape = mean_absolute_percentage_error(y_test, preds)
r2 = r2_score(y_test, preds)

In [9]:
print(f"RMSE: {rmse}")
print(f"MAPE: {mape}")
print(f"R2: {r2}")

RMSE: 0.9933567972727749
MAPE: 0.12099976187347332
R2: 0.5584772694431146
